# Split 70:30 
# Train with RFC and analyse performance
# Compare performance against a two models: 
#    - one that guesses randomly 
#    - another that always return "No".
# Hyperparameter Tuning ?? 5-fold cross-validation.


In [123]:
import os
import re
import random
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.ensemble import RandomForestClassifier

## 1. Load the dataset

In [126]:
path = Path.cwd()
data_path = path.parent / "data"
model_path = path.parent / "models"

# Input: featurised 
input_path = data_path / "csic_2010_featurised.parquet" 

# Output: model
output_path = model_path / "model.joblib" 

In [104]:
# Load the featurised dataset
%time
df_featurised = pd.read_parquet(input_path)

CPU times: user 10 μs, sys: 1 μs, total: 11 μs
Wall time: 26.2 μs


In [105]:
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 20 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   contentLength                  61065 non-null  float64
 1   method_GET                     61065 non-null  float64
 2   method_POST                    61065 non-null  float64
 3   method_PUT                     61065 non-null  float64
 4   host_localhost:8080            61065 non-null  float64
 5   host_localhost:9090            61065 non-null  float64
 6   url_has_select                 61065 non-null  int64  
 7   url_has_drop                   61065 non-null  int64  
 8   url_has_numeric_id             61065 non-null  int64  
 9   url_length                     61065 non-null  int64  
 10  payload_quote_count            61065 non-null  int64  
 11  payload_paren_count            61065 non-null  int64  
 12  payload_slash_count            61065 non-null 

## 2. Split dataset into Training and Test 

Extract training features

In [106]:
numeric_cols = df_featurised.select_dtypes(include=np.number).columns.tolist()

In [107]:
print(len(numeric_cols))
print(numeric_cols)

20
['contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload', 'anomalous']


In [108]:
# All columns are training columns except the last one (`anomalous` which is the target column)
train_cols = numeric_cols[0:-1]
print(len(train_cols))
print(train_cols)

19
['contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload']


In [109]:
# The Last column `anomalous` is the target column
target_col = numeric_cols[-1]
print(len(target_col))
print(target_col)

9
anomalous


In [110]:
# 70:30 Train Test split 
test_size = 0.3
# Const seed
random_state = 42

In [111]:
df_train = df_featurised[train_cols]

In [112]:
df_target = df_featurised[target_col]

In [113]:
#train_inputs

Divides datasets into training and testing subsets.

In [114]:
input_train, input_test, output_train, output_test = train_test_split(
    df_train, # X
    df_target, # y
    test_size=test_size,
    stratify=df_target, # y
    random_state=random_state,
)

> `test_size=0.3` implies `30%` of the dataset for testing, leaving `70%` for training.  
> The dataset is imbalanced, `stratify` ensures that the split maintains the same proportion of classes (`anom` / `norm`) as in the full dataset.
> `input_train` and `output_train` represent the resulting inputs and outputs of the **training data subset** respectively  
> `input_test` and `output_test` represent the resulting the input and output of the **testing data subset** respectively  


In [115]:
input_train

,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,url_has_select,url_has_drop,url_has_numeric_id,url_length,payload_quote_count,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_length,is_17_to_32_bytes,is_empty_payload
38059,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,48,0,0,0,1,0,0,17,0,1
25058,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,50,0,0,0,0,0,0,0,0,1
59118,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,47,0,0,0,0,0,0,0,0,1
52764,71.0,0.0,1.0,0.0,1.0,0.0,0,0,0,48,0,0,0,1,1,1,71,0,0
57737,258.0,0.0,0.0,1.0,0.0,1.0,0,0,0,49,0,0,2,1,6,6,258,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32135,253.0,0.0,1.0,0.0,1.0,0.0,0,0,0,49,0,0,0,1,5,5,253,0,0
51789,34.0,0.0,1.0,0.0,1.0,0.0,0,0,0,47,0,0,0,1,1,1,34,0,0
13760,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,57,0,0,0,1,6,6,70,0,1
23345,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,62,0,0,0,0,0,0,0,0,1


In [116]:
input_test

,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,url_has_select,url_has_drop,url_has_numeric_id,url_length,payload_quote_count,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_length,is_17_to_32_bytes,is_empty_payload
51314,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,48,0,0,0,1,0,0,33,0,1
22432,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,57,0,0,0,0,0,0,0,0,1
949,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,47,0,0,0,0,0,0,0,0,1
51797,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,52,0,0,1,1,1,1,62,0,1
29844,44.0,0.0,1.0,0.0,1.0,0.0,0,0,0,47,0,0,0,1,0,0,44,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144,247.0,0.0,1.0,0.0,1.0,0.0,0,0,0,50,0,0,0,1,2,2,247,0,0
48516,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,47,0,0,0,1,0,0,43,0,1
7441,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,70,0,0,0,0,0,0,0,0,1
34426,0.0,1.0,0.0,0.0,1.0,0.0,0,0,0,44,0,0,0,0,0,0,0,0,1


In [117]:
output_train

38059    0
25058    0
59118    0
52764    0
57737    1
        ..
32135    0
51789    1
13760    1
23345    1
25437    1
Name: anomalous, Length: 42745, dtype: int64

In [118]:
output_test

51314    0
22432    0
949      0
51797    1
29844    1
        ..
2144     0
48516    0
7441     1
34426    0
57290    1
Name: anomalous, Length: 18320, dtype: int64

In [119]:
n_estimators=100
max_depth=10
random_state=42
model = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
)

In [120]:
#model.fit(input_train, output_train)
# Fit the model
model.fit(input_train, output_train.values.ravel())

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap

In [121]:
# feature importances
print("\nFeature Importances:")
for feature, importance in zip(train_cols, model.feature_importances_):
    print(f"    {feature}: {importance:.4f}")


Feature Importances:
    contentLength: 0.0607
    method_GET: 0.0045
    method_POST: 0.0031
    method_PUT: 0.0057
    host_localhost:8080: 0.0058
    host_localhost:9090: 0.0050
    url_has_select: 0.0006
    url_has_drop: 0.0003
    url_has_numeric_id: 0.0486
    url_length: 0.2721
    payload_quote_count: 0.0427
    payload_paren_count: 0.0055
    payload_slash_count: 0.0202
    payload_parameter_count: 0.0392
    payload_percent_count: 0.1514
    payload_percent_encoded_count: 0.1180
    payload_length: 0.2096
    is_17_to_32_bytes: 0.0038
    is_empty_payload: 0.0031


In [137]:
joblib.dump(model, output_path)
print(f"model_saved={model_path}")

model_saved=/mnt/hgfs/workspace/mlhttp/models


In [131]:
preds = model.predict(input_test)
proba = model.predict_proba(input_test)[:, 1]

In [132]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)
from collections import Counter

# Compute metrics
tn, fp, fn, tp = confusion_matrix(output_test, preds).ravel()
precision, recall, _ = precision_recall_curve(output_test, proba)
avg_precision = average_precision_score(output_test, proba)
class_counts = Counter(output_test)

metrics = {
    "accuracy": round(float(accuracy_score(output_test, preds)), 6),
    "precision": round(float(precision_score(output_test, preds, zero_division=0)), 6),
    "recall": round(float(recall_score(output_test, preds, zero_division=0)), 6),
    "f1": round(float(f1_score(output_test, preds, zero_division=0)), 6),
    "roc_auc": round(float(roc_auc_score(output_test, proba)), 6),
    "average_precision": round(float(avg_precision), 6),
    "fpr": round(fp / (fp + tn), 6),  # False Positive Rate
    "fnr": round(fn / (fn + tp), 6),  # False Negative Rate
    "confusion_matrix": confusion_matrix(output_test, preds).tolist(),
    "classification_report": classification_report(output_test, preds, zero_division=0),
    "class_imbalance_ratio": round(class_counts[0] / class_counts[1], 2),  # norm/anom ratio
    "n_test": int(len(input_test)),
}

In [136]:
print("Model Performence metrics")
print(f"Accuracy:          {round(accuracy_score(y_test, preds), 6):.6f}")
print(f"Precision:         {round(precision_score(y_test, preds, zero_division=0), 6):.6f}")
print(f"Recall:            {round(recall_score(y_test, preds, zero_division=0), 6):.6f}")
print(f"F1-Score:          {round(f1_score(y_test, preds, zero_division=0), 6):.6f}")
print(f"ROC-AUC:           {round(roc_auc_score(y_test, proba), 6):.6f}")
print(f"Average Precision: {round(avg_precision, 6):.6f}")
print(f"False Positive Rate: {round(fp / (fp + tn), 6):.6f}")
print(f"False Negative Rate: {round(fn / (fn + tp), 6):.6f}")
print(f"Class Imbalance Ratio (norm/anom): {round(class_counts[0] / class_counts[1], 2):.2f}")
print(f"Test Samples:      {len(X_test)}")

Model Performence metrics
Accuracy:          0.872544
Precision:         0.891321
Recall:            0.785239
F1-Score:          0.834924
ROC-AUC:           0.945730
Average Precision: 0.939055
False Positive Rate: 0.066667
False Negative Rate: 0.214761
Class Imbalance Ratio (norm/anom): 1.44
Test Samples:      18320


# Save the train and test datasets

In [ ]:
df_featurised[numeric_cols].to_parquet(output_path.with_suffix('.parquet'))